# Config Bulk Editor
This notebook allows you to quickly modify parameters across all JSON config files in a directory.

In [1]:
import json
import os
import glob
from pathlib import Path
import copy
from typing import Any, Dict, List

## Configuration

In [2]:
# Set your configs directory path
CONFIGS_DIR = "../configs/wmt"  # Change this to your configs directory path

# Ensure the directory exists
if not os.path.exists(CONFIGS_DIR):
    print(f"Directory {CONFIGS_DIR} does not exist!")
else:
    print(f"Working with configs in: {CONFIGS_DIR}")

Working with configs in: ../configs/wmt


## Helper Functions

In [3]:
def load_all_configs(directory: str) -> Dict[str, Dict]:
    """Load all JSON config files from a directory."""
    configs = {}
    json_files = glob.glob(os.path.join(directory, "*.json"))
    
    for file_path in json_files:
        try:
            with open(file_path, 'r') as f:
                config = json.load(f)
                filename = os.path.basename(file_path)
                configs[filename] = {
                    'path': file_path,
                    'content': config
                }
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
    
    return configs

def get_nested_value(config: Dict, path: str) -> Any:
    """Get value from nested dictionary using dot notation (e.g., 'training.log_every_n_steps')."""
    keys = path.split('.')
    current = config
    
    for key in keys:
        if isinstance(current, dict) and key in current:
            current = current[key]
        else:
            return None
    
    return current

def set_nested_value(config: Dict, path: str, value: Any) -> None:
    """Set value in nested dictionary using dot notation."""
    keys = path.split('.')
    current = config
    
    # Navigate to the parent of the target key
    for key in keys[:-1]:
        if key not in current:
            current[key] = {}
        current = current[key]
    
    # Set the final value
    current[keys[-1]] = value

def save_config(file_path: str, config: Dict) -> None:
    """Save config to file with pretty formatting."""
    with open(file_path, 'w') as f:
        json.dump(config, f, indent=4)

def preview_changes(configs: Dict, changes: Dict[str, Any]) -> None:
    """Preview what changes will be made to each config."""
    print("=== PREVIEW OF CHANGES ===")
    for filename, config_data in configs.items():
        config = config_data['content']
        print(f"\n📁 {filename}:")
        
        for path, new_value in changes.items():
            old_value = get_nested_value(config, path)
            if old_value is not None:
                print(f"  {path}: {old_value} → {new_value}")
            else:
                print(f"  {path}: [NEW] → {new_value}")
    print("\n" + "="*50)

## Load and Inspect Configs

In [4]:
# Load all configs
configs = load_all_configs(CONFIGS_DIR)

print(f"Found {len(configs)} config files:")
for filename in configs.keys():
    print(f"  - {filename}")

Found 14 config files:
  - unsynced_rnn.json
  - transformer.json
  - bidirectional_recurrent_difflogic.json
  - synced_lstm.json
  - bidirectional_unsynced_lstm.json
  - bidirectional_unsynced_gru.json
  - unsynced_gru.json
  - unsynced_lstm.json
  - synced_rnn.json
  - unsynced_recurrent_difflogic.json
  - synced_gru.json
  - bidirectional_unsynced_rnn.json
  - synced_recurrent_difflogic.json
  - synced_feedforward_difflogic.json


## Inspect Current Values

In [5]:
# Check current values for specific parameters
parameters_to_check = [
    "training.log_every_n_steps",
    "training.visualize_every_n_steps",
    "training.epochs",
]

print("=== CURRENT VALUES ===")
for filename, config_data in configs.items():
    config = config_data['content']
    print(f"\n📁 {filename}:")
    
    for param in parameters_to_check:
        value = get_nested_value(config, param)
        print(f"  {param}: {value}")

=== CURRENT VALUES ===

📁 unsynced_rnn.json:
  training.log_every_n_steps: 250
  training.visualize_every_n_steps: 2500
  training.epochs: 1

📁 transformer.json:
  training.log_every_n_steps: 250
  training.visualize_every_n_steps: 2500
  training.epochs: 1

📁 bidirectional_recurrent_difflogic.json:
  training.log_every_n_steps: 250
  training.visualize_every_n_steps: 2500
  training.epochs: 1

📁 synced_lstm.json:
  training.log_every_n_steps: 250
  training.visualize_every_n_steps: 2500
  training.epochs: 1

📁 bidirectional_unsynced_lstm.json:
  training.log_every_n_steps: 250
  training.visualize_every_n_steps: 2500
  training.epochs: 1

📁 bidirectional_unsynced_gru.json:
  training.log_every_n_steps: 250
  training.visualize_every_n_steps: 2500
  training.epochs: 1

📁 unsynced_gru.json:
  training.log_every_n_steps: 250
  training.visualize_every_n_steps: 2500
  training.epochs: 1

📁 unsynced_lstm.json:
  training.log_every_n_steps: 250
  training.visualize_every_n_steps: 2500
  tra

## Define Changes to Make

In [6]:
# Define the changes you want to make
# Format: {"path.to.parameter": new_value}
changes_to_make = {
    "training.log_every_n_steps": 250,
    "training.visualize_every_n_steps": 2500,
    "training.epochs": 1,  # Uncomment to change epochs
    "dataset.params.subset_size":None,
    "tokenizer.params.seq_length":16,
    "tokenizer.params.shared_vocab":True,
    "tokenizer.params.max_vocab_size":16000,
    "tokenizer.params.src_lang":"en",
    "tokenizer.params.tgt_lang":"de",
    "tokenizer.params.max_preprocess_size":500000,
    "tokenizer.params.tokens_per_batch":1024,
    "tokenizer.params.tk_level":"word",
    # "model.params.embedding_dim":256,
    "training.label_smoothing":0.1,
    "scheduler.params.factor":0.95,
    "scheduler.params.patience":10000,
    # "training.optimizer.params.lr": 0.0015, 
    # "training.optimizer.params.weight_decay": 0.001, 
}

print("Changes to be made:")
for path, value in changes_to_make.items():
    print(f"  {path} = {value}")

Changes to be made:
  training.log_every_n_steps = 250
  training.visualize_every_n_steps = 2500
  training.epochs = 1
  dataset.params.subset_size = None
  tokenizer.params.seq_length = 16
  tokenizer.params.shared_vocab = True
  tokenizer.params.max_vocab_size = 16000
  tokenizer.params.src_lang = en
  tokenizer.params.tgt_lang = de
  tokenizer.params.max_preprocess_size = 500000
  tokenizer.params.tokens_per_batch = 1024
  tokenizer.params.tk_level = word
  training.label_smoothing = 0.1
  scheduler.params.factor = 0.95
  scheduler.params.patience = 10000
  training.optimizer.params.lr = 0.0015


## Preview Changes

In [7]:
# Preview what will be changed
preview_changes(configs, changes_to_make)

=== PREVIEW OF CHANGES ===

📁 unsynced_rnn.json:
  training.log_every_n_steps: 250 → 250
  training.visualize_every_n_steps: 2500 → 2500
  training.epochs: 1 → 1
  dataset.params.subset_size: [NEW] → None
  tokenizer.params.seq_length: 16 → 16
  tokenizer.params.shared_vocab: True → True
  tokenizer.params.max_vocab_size: 16000 → 16000
  tokenizer.params.src_lang: en → en
  tokenizer.params.tgt_lang: de → de
  tokenizer.params.max_preprocess_size: 500000 → 500000
  tokenizer.params.tokens_per_batch: 1024 → 1024
  tokenizer.params.tk_level: word → word
  training.label_smoothing: 0.1 → 0.1
  scheduler.params.factor: 0.95 → 0.95
  scheduler.params.patience: 10000 → 10000
  training.optimizer.params.lr: 0.0002 → 0.0015

📁 transformer.json:
  training.log_every_n_steps: 250 → 250
  training.visualize_every_n_steps: 2500 → 2500
  training.epochs: 1 → 1
  dataset.params.subset_size: [NEW] → None
  tokenizer.params.seq_length: 16 → 16
  tokenizer.params.shared_vocab: True → True
  tokenizer.p

## Apply Changes

In [8]:
# WARNING: This will modify your config files!
# Make sure you've backed them up or are using version control

APPLY_CHANGES = True  # Set to True when you're ready to apply changes

if APPLY_CHANGES:
    print("🚀 Applying changes...")
    
    for filename, config_data in configs.items():
        config = config_data['content']
        file_path = config_data['path']
        
        # Make changes
        for path, new_value in changes_to_make.items():
            set_nested_value(config, path, new_value)
        
        # Save the modified config
        save_config(file_path, config)
        print(f"✅ Updated {filename}")
    
    print("\n🎉 All configs updated successfully!")
else:
    print("⚠️  Set APPLY_CHANGES = True to actually modify the files")
    print("⚠️  Make sure to backup your configs first!")

🚀 Applying changes...
✅ Updated unsynced_rnn.json
✅ Updated transformer.json
✅ Updated bidirectional_recurrent_difflogic.json
✅ Updated synced_lstm.json
✅ Updated bidirectional_unsynced_lstm.json
✅ Updated bidirectional_unsynced_gru.json
✅ Updated unsynced_gru.json
✅ Updated unsynced_lstm.json
✅ Updated synced_rnn.json
✅ Updated unsynced_recurrent_difflogic.json
✅ Updated synced_gru.json
✅ Updated bidirectional_unsynced_rnn.json
✅ Updated synced_recurrent_difflogic.json
✅ Updated synced_feedforward_difflogic.json

🎉 All configs updated successfully!


## Quick Templates for Common Changes

In [9]:
# Quick templates for common parameter changes
# Just uncomment and modify the one you need

# Template 1: Logging frequency
# changes_to_make = {
#     "training.log_every_n_steps": 50,
#     "training.visualize_every_n_steps": 500,
# }

# Template 2: Training duration
# changes_to_make = {
#     "training.epochs": 10,
# }

# Template 3: Learning rate adjustments
# changes_to_make = {
#     "training.optimizer.params.lr": 1e-4,
#     "training.optimizer.params.weight_decay": 1e-6,
# }

# Template 4: Dataset parameters
# changes_to_make = {
#     "dataset.params.subset_size": 1000000,
#     "tokenizer.params.seq_length": 64,
# }

# Template 5: Scheduler changes
# changes_to_make = {
#     "training.scheduler.params.patience": 5,
#     "training.scheduler.params.factor": 0.8,
# }

print("Templates ready! Uncomment and modify the one you need.")

Templates ready! Uncomment and modify the one you need.


## Backup and Restore Functions

In [10]:
import shutil
from datetime import datetime

def create_backup(configs_dir: str) -> str:
    """Create a backup of all config files."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_dir = f"{configs_dir}_backup_{timestamp}"
    
    shutil.copytree(configs_dir, backup_dir)
    print(f"✅ Backup created: {backup_dir}")
    return backup_dir

def restore_from_backup(backup_dir: str, configs_dir: str) -> None:
    """Restore configs from backup."""
    if os.path.exists(backup_dir):
        # Remove current configs
        if os.path.exists(configs_dir):
            shutil.rmtree(configs_dir)
        
        # Restore from backup
        shutil.copytree(backup_dir, configs_dir)
        print(f"✅ Restored configs from: {backup_dir}")
    else:
        print(f"❌ Backup directory not found: {backup_dir}")

# Uncomment to create a backup before making changes
# backup_path = create_backup(CONFIGS_DIR)

## Search and Filter Configs

In [11]:
def find_configs_with_parameter(configs: Dict, parameter_path: str, value=None) -> List[str]:
    """Find configs that have a specific parameter, optionally with a specific value."""
    matching_configs = []
    
    for filename, config_data in configs.items():
        config = config_data['content']
        param_value = get_nested_value(config, parameter_path)
        
        if param_value is not None:
            if value is None or param_value == value:
                matching_configs.append(filename)
    
    return matching_configs

# Example: Find configs with specific learning rate
configs_with_lr_1e3 = find_configs_with_parameter(configs, "training.optimizer.params.lr", 1e-3)
print(f"Configs with lr=1e-3: {configs_with_lr_1e3}")

# Example: Find configs that have log_every_n_steps parameter
configs_with_logging = find_configs_with_parameter(configs, "training.log_every_n_steps")
print(f"Configs with log_every_n_steps: {configs_with_logging}")

Configs with lr=1e-3: []
Configs with log_every_n_steps: ['unsynced_rnn.json', 'transformer.json', 'bidirectional_recurrent_difflogic.json', 'synced_lstm.json', 'bidirectional_unsynced_lstm.json', 'bidirectional_unsynced_gru.json', 'unsynced_gru.json', 'unsynced_lstm.json', 'synced_rnn.json', 'unsynced_recurrent_difflogic.json', 'synced_gru.json', 'bidirectional_unsynced_rnn.json', 'synced_recurrent_difflogic.json', 'synced_feedforward_difflogic.json']
